# Fabric Workspace Inventory - semantic-link-labs (Scanner API)

**Purpose:** Same goal as the Core+Admin notebook, but using Microsoft's open-source
`semantic-link-labs` library, which wraps the richer **Scanner API**
(`Admin - WorkspaceInfo`) and adds a governance-focused `list_unused_artifacts()` helper.

**Data sources used:**
1. `sempy_labs.admin.scan_workspaces()` -> Scanner API (createdBy, modifiedBy, createdDate,
   lastUpdatedDate, per-type extended properties)
2. `sempy_labs.admin.list_unused_artifacts()` -> Power BI usage-metrics based "unused" flag
   (covers reports / datasets / dashboards; not lakehouses, warehouses, notebooks, pipelines)
3. Fallback: `sempy.fabric.list_items()` (core Items API, no owner/last-used) if you are not
   a Fabric Administrator

**Permissions required - read before running:**

| Capability | Minimum requirement |
|---|---|
| Install the library (`%pip install`) | Ability to install packages in the Spark session (works by default unless your workspace has restricted outbound network / managed VNet rules) |
| `scan_workspaces()` / `list_unused_artifacts()` | You (or the identity running this notebook) must be a **Fabric Administrator** |
| Fallback basic inventory | Viewer role on the workspace |
| Save snapshot to a Delta table | Contributor+ on the workspace, and a default Lakehouse attached to this notebook |

**If `semantic-link-labs` fails to install** (network restriction, locked-down environment,
no PyPI egress), see the troubleshooting section at the bottom of this notebook - the short
version is: use Notebook 1 instead, which has zero external dependencies beyond `requests`
and `pandas` (both always available in the Fabric runtime).


## 1. Parameters

In [ ]:
# PARAMETERS
workspace_id = ""                              # Leave blank to use the workspace this notebook runs in
save_to_lakehouse = True                      # Set True to append this run's snapshot to a Delta table
lakehouse_table_name =                         "workspace_inventory_scanner_snapshot"
stale_cutoff_days = 90                         # Flag items not modified in more than this many days

## 2. Install semantic-link-labs

This is idempotent - if the library is already present (e.g. added via a custom Fabric
Environment), this cell completes instantly without reinstalling.

In [ ]:
%pip install semantic-link-labs -q

In [ ]:
# Confirm the import actually succeeded before relying on it downstream.
try:
    import sempy_labs as labs
    import sempy_labs.admin as admin
    import sempy.fabric as fabric
    library_available = True
    print("semantic-link-labs imported successfully.")
except ImportError as e:
    library_available = False
    print("Could not import semantic-link-labs after install attempt.")
    print(f"Details: {e}")
    print("See the troubleshooting section at the bottom of this notebook.")

## 3. Setup

In [ ]:
import pandas as pd
from datetime import datetime, timezone
import notebookutils

if not workspace_id:
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

print(f"Target workspace: {workspace_id}")

## 4. Run the Scanner API (falls back to basic inventory if not Fabric Admin)

In [ ]:
scan_available = False
scan_result = None

if library_available:
    try:
        scan_result = admin.scan_workspaces(workspace=workspace_id)
        scan_available = True
        print("Scanner API call succeeded.")
    except Exception as e:
        print("Scanner API failed - this almost always means you are not a Fabric Administrator")
        print("(scan_workspaces requires Fabric Administrator rights, same as the /admin/ REST endpoints).")
        print(f"Details: {e}")
        print("Falling back to sempy.fabric.list_items() for basic inventory (no owner/last-used).")

## 5. Flatten the Scanner JSON into one DataFrame (all item types)

In [ ]:
def extract_items_generic(json_data):
    """
    Flattens the Scanner API JSON into one DataFrame covering every item type present
    in the response (Lakehouse, Warehouse, Notebook, DataPipeline, SemanticModel, Report,
    KQLDatabase, etc.) using the fields common across types.
    """
    rows = []
    skip_keys = {
        "id", "name", "type", "state", "isOnDedicatedCapacity",
        "capacityId", "defaultDatasetStorageFormat"
    }
    for ws in json_data.get("workspaces", []):
        ws_id = ws.get("id")
        ws_name = ws.get("name")
        for key, value in ws.items():
            if key in skip_keys or not isinstance(value, list):
                continue
            for item in value:
                if not isinstance(item, dict):
                    continue
                rows.append({
                    "workspace_id": ws_id,
                    "workspace_name": ws_name,
                    "id": item.get("id"),
                    "name": item.get("name"),
                    "type": key,
                    "description": item.get("description"),
                    "state": item.get("state"),
                    "created_date": item.get("createdDate"),
                    "created_by": item.get("createdBy"),
                    "last_modified": item.get("lastUpdatedDate") or item.get("modifiedDate"),
                    "modified_by": item.get("modifiedBy"),
                })
    return pd.DataFrame(rows)


if scan_available:
    df_final = extract_items_generic(scan_result)
elif library_available:
    fallback_items = fabric.list_items(workspace=workspace_id)
    if not fallback_items.empty:
        fallback_items.columns = [c.strip().lower().replace(" ", "_") for c in fallback_items.columns]
    df_final = fallback_items
else:
    df_final = pd.DataFrame()

if not df_final.empty:
    df_final["snapshot_time_utc"] = datetime.now(timezone.utc).isoformat()
    df_final = df_final.sort_values(["type", "name"]).reset_index(drop=True)

print(f"Total objects found: {len(df_final)}")
display(df_final)

## 6. Object counts by type

In [ ]:
if not df_final.empty and "type" in df_final.columns:
    summary = (
        df_final.groupby("type")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    print("Object counts by type:")
    display(summary)
else:
    print("No data to summarize.")

## 7. Unused artifacts (governance helper)

`list_unused_artifacts()` is based on Power BI usage metrics, so it currently gives the most
reliable signal for **reports, datasets/semantic models, and dashboards**. Lakehouses,
Warehouses, Notebooks, and Pipelines are not covered by this API - use the
`days_since_modified` column from the step below as the practical substitute for those types.

In [ ]:
if scan_available:
    try:
        df_unused = admin.list_unused_artifacts()
        print(f"Found {len(df_unused)} unused artifacts (Power BI usage-metrics based):")
        display(df_unused)
    except Exception as e:
        print("list_unused_artifacts() failed.")
        print(f"Details: {e}")
else:
    print("Skipping unused-artifact check - requires Fabric Administrator / Scanner API access.")

## 8. Flag stale objects by last-modified date

In [ ]:
if not df_final.empty and "last_modified" in df_final.columns and df_final["last_modified"].notna().any():
    df_final["last_modified_dt"] = pd.to_datetime(df_final["last_modified"], errors="coerce", utc=True)
    df_final["days_since_modified"] = (pd.Timestamp.now(tz="UTC") - df_final["last_modified_dt"]).dt.days
    stale_items = df_final[df_final["days_since_modified"] > stale_cutoff_days].sort_values(
        "days_since_modified", ascending=False
    )
    print(f"Items not modified in over {stale_cutoff_days} days ({len(stale_items)} found):")
    display(stale_items[["name", "type", "created_by", "last_modified", "days_since_modified"]])
else:
    print("Staleness check skipped - last_modified data not available (likely not a Fabric Administrator).")

## 9. Optional - persist snapshot to a Delta table for trend tracking

In [ ]:
if save_to_lakehouse and not df_final.empty:
    try:
        spark_df = spark.createDataFrame(df_final.astype(str))
        spark_df.write.mode("append").format("delta").saveAsTable(lakehouse_table_name)
        print(f"Snapshot appended to Delta table: {lakehouse_table_name}")
    except Exception as e:
        print("Could not save to lakehouse - make sure a default Lakehouse is attached to this notebook.")
        print(f"Details: {e}")
else:
    print("save_to_lakehouse=False (or no data) - skipping persistence.")

## Permissions recap

- **Installing the library:** just needs Spark session package-install ability. Fails only if
  your workspace/capacity has restricted outbound network rules (managed VNet without PyPI
  in the allow-list) or a locked-down custom Environment.
- **`scan_workspaces()` / `list_unused_artifacts()`:** the identity running this notebook must
  be a **Fabric Administrator** - same requirement as the raw `/v1/admin/*` REST endpoints,
  since these functions call them internally.
- **Fallback `fabric.list_items()`:** Viewer role on the workspace, no admin needed.
- **Saving to Delta:** Contributor+ role on the workspace, plus a default Lakehouse attached
  to this notebook with write access.

## Troubleshooting - "semantic-link-labs won't install"

1. **Check it's actually a network/permission issue, not a typo** - re-run the install cell
   and read the actual pip error rather than assuming.
2. **Restricted/managed VNet workspace:** ask your Fabric/network admin to allow-list
   `pypi.org`, `files.pythonhosted.org`, and `pythonhosted.org` in the workspace's outbound
   access rules. This is the most common cause in enterprise tenants.
3. **Offline install via a wheel file:** download the `.whl` from
   [PyPI](https://pypi.org/project/semantic-link-labs/) or
   [GitHub releases](https://github.com/microsoft/semantic-link-labs/releases) on a machine
   that does have internet, upload it to the notebook's **Resources** pane (not the Files
   section), then run `%pip install "<relative path to the .whl>"`.
4. **Make it permanent (recommended for production):** create a custom Fabric **Environment**,
   add `semantic-link-labs` as a public library there once, publish it, and attach that
   Environment to this notebook. This avoids a `pip install` on every run and is faster and
   more reliable for scheduled jobs.
5. **Still blocked?** Fall back to **Notebook 1** (Core + Admin REST API). It only needs
   `requests` and `pandas`, both of which ship with every Fabric runtime by default - no
   external package installation required at all.


In [ ]:
%%sql
select * from LK_Fabric.workspace_inventory_scanner_snapshot